## EDA: 데이터 구조 파악

In [ ]:
import csv
from collections import Counter
import json
import statistics
from collections import defaultdict, Counter

### Step 1: 클래스 분포

In [ ]:
# 1. read train csv
labels = []
for row in csv.DictReader(open("../data/train_labels.csv", encoding="utf-8")):
    labels.append(row["action"])

# 2. count labels
counter = Counter(labels)
total = len(labels)

# 3. print
for action, cnt in sorted(counter.items(), key=lambda x: -x[1]):
    print(f"{action}: {cnt}건 ({cnt/total:.2%})")

edit_file: 11171건 (15.96%)
grep_search: 9912건 (14.16%)
read_file: 9257건 (13.22%)
glob_pattern: 5284건 (7.55%)
respond_only: 5178건 (7.40%)
run_bash: 5068건 (7.24%)
apply_patch: 4823건 (6.89%)
run_tests: 4561건 (6.52%)
list_directory: 4329건 (6.18%)
ask_user: 2701건 (3.86%)
plan_task: 2679건 (3.83%)
lint_or_typecheck: 2283건 (3.26%)
write_file: 1481건 (2.12%)
web_search: 1273건 (1.82%)


### Step 2: current_prompt 길이 분포

In [5]:
# 1. read train csv
labels = {}
for row in csv.DictReader(open('../data/train_labels.csv', encoding='utf-8')):
    labels[row['id']] = row['action']

# 2. read train jsonl
samples = []
for line in open('../data/train.jsonl', encoding='utf-8'):
    if line.strip():
        samples.append(json.loads(line))

# 3. prompt lenths by action
action_lens = defaultdict(list)
for s in samples:
    action = labels[s['id']]
    length = len(s['current_prompt'])
    action_lens[action].append(length)

# 4. total statics
all_lens = [len(s['current_prompt']) for s in samples]
print(f'전체 평균: {statistics.mean(all_lens):.1f}자')
print(f'전체 최소: {min(all_lens)}자')
print(f'전체 최대: {max(all_lens)}자')
print()

# 5. print avg lenth by action
for action, lens in sorted(action_lens.items(), key=lambda x: -statistics.mean(x[1])):
    print(f'{action:<25} 평균 {statistics.mean(lens):.1f}자')


전체 평균: 61.0자
전체 최소: 2자
전체 최대: 346자

plan_task                 평균 87.6자
ask_user                  평균 85.8자
web_search                평균 85.2자
write_file                평균 72.2자
apply_patch               평균 66.2자
edit_file                 평균 65.6자
glob_pattern              평균 64.7자
grep_search               평균 64.4자
list_directory            평균 64.1자
read_file                 평균 62.9자
respond_only              평균 42.9자
run_tests                 평균 40.1자
lint_or_typecheck         평균 39.5자
run_bash                  평균 39.2자


| 그룹 | action | 평균 길이 |
|------|--------|-----------|
| 긴 그룹 | plan_task, ask_user, web_search | 85~88자 |
| 짧은 그룹 | run_bash, lint_or_typecheck, run_tests | 39~43자 |

**인사이트**
- 전체 평균 61자, 최소 2자, 최대 346자
- 짧은 그룹과 긴 그룹이 약 2배 차이로 명확히 구분됨
- → current_prompt 길이를 피처로 추가 예정

### Step 3: history 패턴 분석
직전 행동이 다음 행동을 예측하는 데 도움이 되는지 확인

In [7]:
# 1. history 길이 통계
history_lens = [len(s['history']) for s in samples]
print(f'history 평균 턴 수: {statistics.mean(history_lens):.1f}')
print(f'history 0인 샘플 (첫 턴): {sum(1 for h in history_lens if h == 0)}건')
print()

# 2. 첫 턴일 때 action 분포
first_turn = [labels[s['id']] for s in samples if len(s['history']) == 0]
print('=== 첫 턴일 때 action 분포 ===')
for action, cnt in Counter(first_turn).most_common():
    print(f'  {action:<25} {cnt}건')
print()

# 3. 직전 action → 현재 action 전이 패턴
transition = defaultdict(list)
for s in samples:
    curr = labels[s['id']]
    prev_actions = [h['name'] for h in s['history'] if h['role'] == 'assistant_action']
    if prev_actions:
        transition[prev_actions[-1]].append(curr)

print('=== 직전 action → 다음 action (상위 3개) ===')
for prev, nexts in sorted(transition.items()):
    top3 = Counter(nexts).most_common(3)
    top3_str = ', '.join(f'{a}({c}건)' for a, c in top3)
    print(f'  {prev:<25} → {top3_str}')

history 평균 턴 수: 6.9
history 0인 샘플 (첫 턴): 9000건

=== 첫 턴일 때 action 분포 ===
  list_directory            1818건
  read_file                 1488건
  plan_task                 1121건
  grep_search               1046건
  run_bash                  1011건
  write_file                708건
  ask_user                  629건
  glob_pattern              567건
  edit_file                 228건
  run_tests                 120건
  web_search                109건
  respond_only              77건
  lint_or_typecheck         72건
  apply_patch               6건

=== 직전 action → 다음 action (상위 3개) ===
  apply_patch               → lint_or_typecheck(757건), respond_only(713건), apply_patch(689건)
  ask_user                  → grep_search(439건), read_file(335건), edit_file(309건)
  edit_file                 → run_tests(2444건), edit_file(1638건), apply_patch(1076건)
  glob_pattern              → grep_search(1102건), glob_pattern(872건), read_file(771건)
  grep_search               → edit_file(2065건), read_file(1819건), grep_search(1

### Step 4: session_meta feature 분석

In [ ]:
# 1. user_tier별 action 분포 (요금제에 따른 action 패턴)
print('=== user_tier별 상위 5개 action ===')
for tier in ['enterprise', 'pro', 'free']:
    actions = [labels[s['id']] for s in samples
               if s['session_meta']['user_tier'] == tier]
    top5 = Counter(actions).most_common(5)
    print(f'{tier}: {top5}')
print()

# 2. last_ci_status별 action 분포 (현재 작업환경 상태)
print('=== last_ci_status별 상위 5개 action ===')
for status in ['passed', 'failed', 'none']:
    actions = [labels[s['id']] for s in samples
               if s['session_meta']['workspace']['last_ci_status'] == status]
    top5 = Counter(actions).most_common(5)
    print(f'CI {status}: {top5}')
print()

# 3. git_dirty별 action 분포 (변경사항 상태)
print('=== git_dirty별 상위 5개 action ===')
for dirty in [True, False]:
    actions = [labels[s['id']] for s in samples
               if s['session_meta']['workspace']['git_dirty'] == dirty]
    top5 = Counter(actions).most_common(5)
    print(f'git_dirty={dirty}: {top5}')
print()

# 4. turn_index 분포 (세션 진행 단계)
print('=== turn_index 통계 ===')
turn_indices = [s['session_meta']['turn_index'] for s in samples]
print(f'평균: {statistics.mean(turn_indices):.1f}')
print(f'최소: {min(turn_indices)}')
print(f'최대: {max(turn_indices)}')

=== user_tier별 상위 5개 action ===
enterprise: [('edit_file', 1802), ('read_file', 1617), ('grep_search', 1550), ('run_bash', 839), ('apply_patch', 826)]
pro: [('edit_file', 5944), ('grep_search', 5448), ('read_file', 4944), ('glob_pattern', 2948), ('respond_only', 2801)]
free: [('edit_file', 3425), ('grep_search', 2914), ('read_file', 2696), ('respond_only', 1562), ('run_bash', 1552)]

=== last_ci_status별 상위 5개 action ===
CI passed: [('edit_file', 4261), ('grep_search', 3826), ('read_file', 3820), ('respond_only', 3062), ('glob_pattern', 1997)]
CI failed: [('edit_file', 4067), ('grep_search', 3173), ('read_file', 2743), ('apply_patch', 2189), ('run_tests', 1811)]
CI none: [('grep_search', 2913), ('edit_file', 2843), ('read_file', 2694), ('glob_pattern', 1654), ('run_bash', 1567)]

=== git_dirty별 상위 5개 action ===
git_dirty=True: [('edit_file', 8762), ('grep_search', 7372), ('read_file', 6139), ('apply_patch', 4570), ('respond_only', 4551)]
git_dirty=False: [('read_file', 3118), ('grep_sea

### Step 5: 언어별 분포

In [9]:
# language_pref별 action 분포
print('=== language_pref별 상위 5개 action ===')
for lang in ['ko', 'en', 'mixed']:
    actions = [labels[s['id']] for s in samples
               if s['session_meta']['language_pref'] == lang]
    total_lang = len(actions)
    top5 = Counter(actions).most_common(5)
    print(f'\n{lang} ({total_lang}건)')
    for action, cnt in top5:
        print(f'  {action:<25} {cnt}건  {cnt/total_lang*100:.1f}%')

=== language_pref별 상위 5개 action ===

ko (45028건)
  edit_file                 7243건  16.1%
  grep_search               6326건  14.0%
  read_file                 5913건  13.1%
  respond_only              3357건  7.5%
  glob_pattern              3343건  7.4%

en (17802건)
  edit_file                 2773건  15.6%
  grep_search               2572건  14.4%
  read_file                 2350건  13.2%
  glob_pattern              1435건  8.1%
  respond_only              1323건  7.4%

mixed (7170건)
  edit_file                 1155건  16.1%
  grep_search               1014건  14.1%
  read_file                 994건  13.9%
  run_bash                  562건  7.8%
  glob_pattern              506건  7.1%
